# nb_00_generate_mock_data

**Purpose**: Generate realistic mock insurance data for 7 domain tables using Python/Faker.

**Outputs**: CSV files written directly to the Bronze Lakehouse at `Files/seed-data/`:
- `customers.csv` (750 rows)
- `agents.csv` (50 rows)
- `policies.csv` (1000 rows)
- `coverages.csv` (2000 rows)
- `premiums.csv` (3000 rows)
- `claims.csv` (800 rows)
- `claim_payments.csv` (500 rows)

**Data Quality**: ~2% of rows intentionally contain bad data (null required fields, duplicate business keys) to demonstrate Silver-layer quality checks.

**Note**: This notebook must run in Fabric with `lh_bronze_insurance` set as the Default Lakehouse. CSVs are written to OneLake so they are immediately available for `nb_01_bronze_ingest`.

In [ ]:
import os
import csv
import random
import uuid
from datetime import datetime, timedelta

# Seed for reproducibility
random.seed(42)

# Output directory — writes to Bronze Lakehouse Files section
# When lh_bronze_insurance is set as Default Lakehouse, /lakehouse/default maps to it
DATA_DIR = "/lakehouse/default/Files/seed-data"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Output directory: {DATA_DIR}")

In [ ]:
# ============================================================
# Helper functions
# ============================================================

FIRST_NAMES = [
    "James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda",
    "David", "Elizabeth", "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica",
    "Thomas", "Sarah", "Christopher", "Karen", "Charles", "Lisa", "Daniel", "Nancy",
    "Matthew", "Betty", "Anthony", "Margaret", "Mark", "Sandra", "Donald", "Ashley",
    "Steven", "Dorothy", "Andrew", "Kimberly", "Paul", "Emily", "Joshua", "Donna",
    "Kenneth", "Michelle", "Kevin", "Carol", "Brian", "Amanda", "George", "Melissa",
    "Timothy", "Deborah", "Ronald", "Stephanie", "Edward", "Rebecca", "Jason", "Sharon",
    "Jeffrey", "Laura", "Ryan", "Cynthia"
]

LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis",
    "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson",
    "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson",
    "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker",
    "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores",
    "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell",
    "Carter", "Roberts"
]

STREETS = ["Main St", "Oak Ave", "Elm St", "Pine Rd", "Maple Dr", "Cedar Ln", "Birch Ct",
           "Walnut St", "Cherry Blvd", "Spruce Way", "Park Ave", "Lake Dr", "Hill Rd",
           "River Rd", "Forest Ln", "Meadow Dr", "Valley Rd", "Spring St", "Summit Ave"]

CITIES = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia",
          "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville",
          "Fort Worth", "Columbus", "Charlotte", "Indianapolis", "Denver", "Seattle",
          "Portland", "Nashville"]

STATES = ["NY", "CA", "IL", "TX", "AZ", "PA", "TX", "CA", "TX", "CA",
          "TX", "FL", "TX", "OH", "NC", "IN", "CO", "WA", "OR", "TN"]

REGIONS = ["Northeast", "Southeast", "Midwest", "Southwest", "West", "Pacific Northwest"]

POLICY_TYPES = ["auto", "home", "life", "health"]
POLICY_STATUSES = ["active", "expired", "cancelled"]
CLAIM_STATUSES = ["open", "under_review", "approved", "denied", "closed"]
CLAIM_TYPES = ["collision", "comprehensive", "liability", "property_damage", "medical", "theft", "fire", "flood"]
PAYMENT_METHODS = ["check", "ach", "wire_transfer", "direct_deposit"]
PAYMENT_STATUSES = ["paid", "pending", "overdue", "partial"]
COVERAGE_TYPES = ["liability", "collision", "comprehensive", "uninsured_motorist",
                  "dwelling", "personal_property", "medical_payments", "term_life",
                  "whole_life", "hospitalization", "prescription", "dental", "vision"]

def random_date(start_year, end_year):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = (end - start).days
    return (start + timedelta(days=random.randint(0, delta))).strftime("%Y-%m-%d")

def random_phone():
    return f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}"

def random_email(first, last):
    domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "aol.com", "proton.me"]
    return f"{first.lower()}.{last.lower()}{random.randint(1,999)}@{random.choice(domains)}"

def random_zip():
    return f"{random.randint(10000, 99999)}"

def random_license():
    return f"LIC-{random.randint(100000, 999999)}"

def write_csv(filepath, headers, rows):
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)
    print(f"  Written {len(rows)} rows to {os.path.basename(filepath)}")

In [ ]:
# ============================================================
# 1. CUSTOMERS (750 rows, ~2% bad)
# ============================================================
print("Generating customers...")
NUM_CUSTOMERS = 750
BAD_CUSTOMER_COUNT = int(NUM_CUSTOMERS * 0.02)

customers = []
customer_ids = []

for i in range(1, NUM_CUSTOMERS + 1):
    cid = f"CUST-{i:05d}"
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    city_idx = random.randint(0, len(CITIES) - 1)
    
    row = [
        cid,
        first,
        last,
        random_email(first, last),
        random_phone(),
        random_date(1950, 2000),
        f"{random.randint(100, 9999)} {random.choice(STREETS)}",
        CITIES[city_idx],
        STATES[city_idx],
        random_zip(),
        random_date(2010, 2025)
    ]
    customers.append(row)
    customer_ids.append(cid)

# Seed bad rows: null first_name, null email
for i in random.sample(range(len(customers)), BAD_CUSTOMER_COUNT):
    if random.random() < 0.5:
        customers[i][1] = ""  # null first_name
    else:
        customers[i][3] = ""  # null email

# Seed duplicate business keys (2 duplicates)
for _ in range(2):
    dup = list(customers[random.randint(0, len(customers) - 1)])
    customers.append(dup)

random.shuffle(customers)

write_csv(
    os.path.join(DATA_DIR, "customers.csv"),
    ["customer_id", "first_name", "last_name", "email", "phone", "date_of_birth",
     "address", "city", "state", "zip_code", "customer_since"],
    customers
)

In [ ]:
# ============================================================
# 2. AGENTS (50 rows, ~2% bad)
# ============================================================
print("Generating agents...")
NUM_AGENTS = 50

agents = []
agent_ids = []

for i in range(1, NUM_AGENTS + 1):
    aid = f"AGT-{i:04d}"
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    row = [
        aid,
        first,
        last,
        random.choice(REGIONS),
        random_license(),
        random_date(2005, 2023),
        random.choice(["active", "active", "active", "inactive"])  # 75% active
    ]
    agents.append(row)
    agent_ids.append(aid)

# Seed 1 bad row: null region
agents[random.randint(0, len(agents) - 1)][3] = ""

write_csv(
    os.path.join(DATA_DIR, "agents.csv"),
    ["agent_id", "first_name", "last_name", "region", "license_number", "hire_date", "status"],
    agents
)

In [ ]:
# ============================================================
# 3. POLICIES (1000 rows, ~2% bad)
# ============================================================
print("Generating policies...")
NUM_POLICIES = 1000
BAD_POLICY_COUNT = int(NUM_POLICIES * 0.02)

policies = []
policy_ids = []

for i in range(1, NUM_POLICIES + 1):
    pid = f"POL-{i:06d}"
    eff_year = random.randint(2018, 2025)
    eff_date = random_date(eff_year, eff_year)
    exp_date = (datetime.strptime(eff_date, "%Y-%m-%d") + timedelta(days=365)).strftime("%Y-%m-%d")
    ptype = random.choice(POLICY_TYPES)
    
    premium_ranges = {"auto": (800, 3000), "home": (1200, 5000), "life": (500, 2500), "health": (3000, 12000)}
    low, high = premium_ranges[ptype]
    
    row = [
        pid,
        random.choice(customer_ids),
        random.choice(agent_ids),
        ptype,
        eff_date,
        exp_date,
        random.choices(POLICY_STATUSES, weights=[60, 25, 15])[0],
        round(random.uniform(low, high), 2)
    ]
    policies.append(row)
    policy_ids.append(pid)

# Seed bad rows: null policy_type, null customer_id
for i in random.sample(range(len(policies)), BAD_POLICY_COUNT):
    if random.random() < 0.5:
        policies[i][3] = ""  # null policy_type
    else:
        policies[i][1] = ""  # null customer_id

# Seed 3 duplicate policy_ids
for _ in range(3):
    dup = list(policies[random.randint(0, len(policies) - 1)])
    policies.append(dup)

random.shuffle(policies)

write_csv(
    os.path.join(DATA_DIR, "policies.csv"),
    ["policy_id", "customer_id", "agent_id", "policy_type", "effective_date",
     "expiration_date", "status", "annual_premium"],
    policies
)

In [ ]:
# ============================================================
# 4. COVERAGES (2000 rows, ~2% bad)
# ============================================================
print("Generating coverages...")
NUM_COVERAGES = 2000
BAD_COVERAGE_COUNT = int(NUM_COVERAGES * 0.02)

coverages = []
coverage_ids = []

for i in range(1, NUM_COVERAGES + 1):
    cov_id = f"COV-{i:06d}"
    row = [
        cov_id,
        random.choice(policy_ids),
        random.choice(COVERAGE_TYPES),
        round(random.uniform(10000, 500000), 2),
        round(random.uniform(250, 5000), 2)
    ]
    coverages.append(row)
    coverage_ids.append(cov_id)

# Seed bad rows: null coverage_type
for i in random.sample(range(len(coverages)), BAD_COVERAGE_COUNT):
    coverages[i][2] = ""  # null coverage_type

write_csv(
    os.path.join(DATA_DIR, "coverages.csv"),
    ["coverage_id", "policy_id", "coverage_type", "coverage_limit", "deductible_amount"],
    coverages
)

In [ ]:
# ============================================================
# 5. PREMIUMS (3000 rows, ~2% bad)
# ============================================================
print("Generating premiums...")
NUM_PREMIUMS = 3000
BAD_PREMIUM_COUNT = int(NUM_PREMIUMS * 0.02)

premiums = []
premium_ids = []

for i in range(1, NUM_PREMIUMS + 1):
    prem_id = f"PREM-{i:06d}"
    amount_due = round(random.uniform(50, 1500), 2)
    status = random.choices(PAYMENT_STATUSES, weights=[60, 20, 10, 10])[0]
    
    if status == "paid":
        amount_paid = amount_due
    elif status == "partial":
        amount_paid = round(amount_due * random.uniform(0.2, 0.8), 2)
    else:
        amount_paid = 0.0
    
    pay_date = random_date(2020, 2025) if status in ["paid", "partial"] else ""
    billing_period = random.choice(["2020-Q1", "2020-Q2", "2020-Q3", "2020-Q4",
                                    "2021-Q1", "2021-Q2", "2021-Q3", "2021-Q4",
                                    "2022-Q1", "2022-Q2", "2022-Q3", "2022-Q4",
                                    "2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4",
                                    "2024-Q1", "2024-Q2", "2024-Q3", "2024-Q4",
                                    "2025-Q1", "2025-Q2", "2025-Q3", "2025-Q4"])
    row = [
        prem_id,
        random.choice(policy_ids),
        billing_period,
        amount_due,
        amount_paid,
        pay_date,
        status
    ]
    premiums.append(row)
    premium_ids.append(prem_id)

# Seed bad rows: null amount_due
for i in random.sample(range(len(premiums)), BAD_PREMIUM_COUNT):
    premiums[i][3] = ""  # null amount_due

# Seed 3 duplicate premium_ids
for _ in range(3):
    dup = list(premiums[random.randint(0, len(premiums) - 1)])
    premiums.append(dup)

random.shuffle(premiums)

write_csv(
    os.path.join(DATA_DIR, "premiums.csv"),
    ["premium_id", "policy_id", "billing_period", "amount_due", "amount_paid",
     "payment_date", "payment_status"],
    premiums
)

In [ ]:
# ============================================================
# 6. CLAIMS (800 rows, ~2% bad)
# ============================================================
print("Generating claims...")
NUM_CLAIMS = 800
BAD_CLAIM_COUNT = int(NUM_CLAIMS * 0.02)

claims = []
claim_ids = []
approved_or_closed_claim_ids = []

CLAIM_DESCRIPTIONS = [
    "Rear-end collision on highway", "Water damage from burst pipe",
    "Hail damage to roof", "Vehicle theft from parking lot",
    "Kitchen fire due to electrical fault", "Slip and fall on property",
    "Windshield cracked by debris", "Flood damage to basement",
    "Fender bender in parking garage", "Tree fell on house during storm",
    "Medical emergency requiring hospitalization", "Vandalism to parked vehicle",
    "Smoke damage from neighboring unit", "Stolen personal electronics",
    "Dog bite liability claim", "Minor collision at intersection"
]

for i in range(1, NUM_CLAIMS + 1):
    clm_id = f"CLM-{i:06d}"
    loss_date = random_date(2020, 2025)
    filed_date = (datetime.strptime(loss_date, "%Y-%m-%d") + timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%d")
    status = random.choices(CLAIM_STATUSES, weights=[15, 20, 30, 15, 20])[0]
    
    row = [
        clm_id,
        random.choice(policy_ids),
        loss_date,
        filed_date,
        random.choice(CLAIM_TYPES),
        status,
        round(random.uniform(500, 75000), 2),
        random.choice(CLAIM_DESCRIPTIONS)
    ]
    claims.append(row)
    claim_ids.append(clm_id)
    if status in ["approved", "closed"]:
        approved_or_closed_claim_ids.append(clm_id)

# Seed bad rows: null claim_status, null policy_id
for i in random.sample(range(len(claims)), BAD_CLAIM_COUNT):
    if random.random() < 0.5:
        claims[i][5] = ""  # null claim_status
    else:
        claims[i][1] = ""  # null policy_id

# Seed 2 duplicate claim_ids
for _ in range(2):
    dup = list(claims[random.randint(0, len(claims) - 1)])
    claims.append(dup)

random.shuffle(claims)

write_csv(
    os.path.join(DATA_DIR, "claims.csv"),
    ["claim_id", "policy_id", "date_of_loss", "date_filed", "claim_type",
     "claim_status", "estimated_amount", "description"],
    claims
)

In [ ]:
# ============================================================
# 7. CLAIM PAYMENTS (500 rows, ~2% bad)
# ============================================================
print("Generating claim payments...")
NUM_CLAIM_PAYMENTS = 500
BAD_PAYMENT_COUNT = int(NUM_CLAIM_PAYMENTS * 0.02)

# Use only approved/closed claims as FK references
source_claims = approved_or_closed_claim_ids if len(approved_or_closed_claim_ids) > 0 else claim_ids

claim_payments = []

for i in range(1, NUM_CLAIM_PAYMENTS + 1):
    pay_id = f"CPAY-{i:06d}"
    row = [
        pay_id,
        random.choice(source_claims),
        random_date(2020, 2025),
        round(random.uniform(200, 50000), 2),
        random.choice(PAYMENT_METHODS)
    ]
    claim_payments.append(row)

# Seed bad rows: null payment_amount
for i in random.sample(range(len(claim_payments)), BAD_PAYMENT_COUNT):
    claim_payments[i][3] = ""  # null payment_amount

# Seed 2 duplicate payment_ids
for _ in range(2):
    dup = list(claim_payments[random.randint(0, len(claim_payments) - 1)])
    claim_payments.append(dup)

random.shuffle(claim_payments)

write_csv(
    os.path.join(DATA_DIR, "claim_payments.csv"),
    ["payment_id", "claim_id", "payment_date", "payment_amount", "payment_method"],
    claim_payments
)

In [ ]:
# ============================================================
# VALIDATION SUMMARY
# ============================================================
print("\n" + "="*60)
print("MOCK DATA GENERATION COMPLETE")
print("="*60)

for fname in ["customers.csv", "agents.csv", "policies.csv", "coverages.csv",
              "premiums.csv", "claims.csv", "claim_payments.csv"]:
    fpath = os.path.join(DATA_DIR, fname)
    with open(fpath, "r", encoding="utf-8") as f:
        row_count = sum(1 for _ in f) - 1  # subtract header
    print(f"  {fname:<25} {row_count:>6} rows")

print(f"\nAll files saved to: {DATA_DIR}")
print("CSVs are in the Bronze Lakehouse at Files/seed-data/ — ready for nb_01_bronze_ingest.")
print("~2% of rows contain intentional bad data for Silver-layer demo.")